# Get word embeddings using Word2vec

## Introduction

In this tutorial we'll get vector representations of words (word embeddings) with word2vec algorithms using gensim.
We'll show you how to get words embeddings the easy way if the corpus is small, and what to do if the size of the corpus doesn't allow you to download the entire corpus in RAM.

The goal of this tutorial is to show the word2vec so you can start using it yourself.

## Requirements

- Python 3
- gensim 4.0.0
- downloaded corpus (follow the link to download https://shortlinks.plus/gZteg)
- 4 GB RAM
- 8 GB disk space for WMD

## Simple example for Russian

### 0. Before we get started

First of all, read the corpus line-by-line (new line = new document) and get a list of lists, where the elements of the external list are documents and the elements of the document are words that occur in this document

In [ ]:
import io

def load_corpus(fname):
    fin = io.open(fname, 'r', encoding='utf-8', newline='\n', errors='ignore')
    documents = []
    for line in fin:
        documents.append(line.split())
    return documents

Next, we need save dictionary in file and load it from file

In [ ]:
def save_dictionary(fname, dictionary, args):
    length, dimension = args
    fin = io.open(fname, 'w', encoding='utf-8')
    fin.write('%d %d\n' % (length, dimension))
    for word in dictionary:
        fin.write('%s %s\n' % (word, ' '.join(map(str, dictionary[word]))))

def load_dictionary(fname):
    fin = io.open(fname, 'r', encoding='utf-8', newline='\n', errors='ignore')
    length, dimension = map(int, fin.readline().split())
    dictionary = {}
    for line in fin:
        tokens = line.rstrip().split(' ')
        dictionary[tokens[0]] = map(float, tokens[1:])
    return dictionary

### 1. Load our corpus

In [ ]:
documents = load_corpus('russian_lit_corpus.txt')

For checking, the corpus contains 12692 documents

In [ ]:
len(documents)

### 2. Train word2vec model

You can tunning this model yourself:
- *vector_size*: dimensionality of the word vectors (default 100)
- *window*: maximum distance between the current and predicted word within a sentence (default 5)
- *min_count*: ignores all words with total frequency lower than this (default 5)
- *workers*: use these many worker threads to train the model (default 3)
- *sg*: training algorithm; 1 for skip-gram, otherwise CBOW  (default 0)
- *hs*: if 1, hierarchical softmax will be used for model training; if 0, and negative is non-zero, negative sampling will be used (default 0)
- *negative*: if > 0, negative sampling will be used, the int for negative specifies how many “noise words” should be drawn (usually between 5-20); if set to 0, no negative sampling is used (default 5)
- *max_vocab_size*: limits the RAM during vocabulary building; if there are more unique words than this, then prune the infrequent ones; every 10 million word types need about 1GB of RAM; set to None for no limit (default None)
- *epochs*: number of iterations (epochs) over the corpus (default 5)

Read more: https://radimrehurek.com/gensim/models/word2vec.html

In [ ]:
%%time
from gensim.models import Word2Vec

dimension = 8
model = Word2Vec(sentences=documents, vector_size=dimension, min_count=1)

In [ ]:
dictionary = {key : model.wv[key] for key in model.wv.key_to_index}

For checking, the dictionary contains 192881 different words (if min_count = 1)

In [ ]:
len(dictionary)

Using word2vec models, you can find the closest word. For instance, I test the quality of the model on abstract nouns

In [ ]:
model.wv.most_similar('любовь')

### 3. Save dictionary in file

In [ ]:
save_dictionary('russian_lit_dictionary.txt', dictionary, (len(dictionary), dimension))

### 4. Check that everything is saved correctly (optional)

In [ ]:
loaded_dictionary = load_dictionary('russian_lit_dictionary.txt')
len(dictionary) == len(loaded_dictionary)

## Memory error or example for English with enormous corpus

Oops.. If you tried to run the previous code and got a Memory Error, putting the whole corpus into RAM was a bad idea. Using the English corpus as an example, let's reveal another advantage of word2vec: if we save the model we can continue training it later

### 0. Before we get started

To retrain a model, we first need to build the vocabulary we are using

In [ ]:
def get_vocab(fname):
    fin = io.open(fname, 'r', encoding='utf-8', newline='\n', errors='ignore')
    vocab = set()
    for line in fin:
        for word in line.split():
            vocab.add(word)
    return vocab

### 1. Get vocabulary

In [ ]:
vocab = get_vocab('english_lit_corpus.txt')

For checking, the corpus contains 315436 words

In [ ]:
len(vocab)

### 2. Create model and build vocabulary

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(vector_size=8, min_count=1)
model.build_vocab(vocab)
model.save('saved_model')

### 3. Train word2vec model

In [ ]:
%%time

with io.open('english_lit_corpus.txt', 'r', encoding='utf-8', newline='\n', errors='ignore') as file:
    eof = False
    while not eof:
        limit = 1000
        documents = []
        for line in file:
            documents.append(line.split())
            limit -= 1
            if limit == 0:
                break
        else:
            eof = True
        model = Word2Vec.load('saved_model')
        model.build_vocab(documents, update=True)
        model.train(documents, total_examples=model.corpus_count, epochs=model.epochs)
        model.save('saved_model')

In [ ]:
dictionary = {key : model.wv[key] for key in model.wv.key_to_index}

For checking, the dictionary contains 315480 different words (if min_count = 1)

In [ ]:
len(dictionary)

### 4. Save dictionary in file

In [ ]:
save_dictionary('english_lit_dictionary.txt', dictionary, (len(dictionary), 8))

## A bit about pretrained word embedding models

Pre-trained models are ready-to-use models, usually trained on large corpus and containing millions of words. One of the best models belongs to FastText.

FastText is an open-source, free, lightweight library that allows users to learn text representations and text classifiers. It works on standard, generic hardware. Models can later be reduced in size to even fit on mobile devices.

### 0. Before we get started, choose a model

### Common Crawl word vectors

**Link:** https://fasttext.cc/docs/en/crawl-vectors.html

**Algorithm**

CBOW with position-weights, in dimension 300, with character n-grams of length 5, a window of size 5 and 10 negatives.

**Training data**
- Wikipedia,
- Common Crawl

Wikipedia is the largest free online encyclopedia, available in more than 200 different languages; because the articles are curated, the corresponding text is of high quality, making Wikipedia a great resource for (multilingual) NLP. Common Crawl is a non profit organization which crawls the web and makes the resulting data publicly available.

Read more: https://arxiv.org/pdf/1802.06893.pdf

**Languages**

Afrikaans, Albanian, Alemannic, Amharic, Arabic, Aragonese, Armenian, Assamese, Asturian, Azerbaijani, Bashkir, Basque, Bavarian, Belarusian, Bengali, Bihari, Bishnupriya Manipuri, Bosnian, Breton, Bulgarian, Burmese, Catalan, Cebuano, Central Bicolano, Chechen, Chinese, Chuvash, Corsican, Croatian, Czech, Danish, Divehi, Dutch, Eastern Punjabi, Egyptian Arabic, Emilian-Romagnol, English, Erzya, Esperanto, Estonian, Fiji Hindi, Finnish, French, Galician, Georgian, German, Goan Konkani, Greek, Gujarati, Haitian, Hebrew, Hill Mari, Hindi, Hungarian, Icelandic, Ido, Ilokano, Indonesian, Interlingua, Irish, Italian, Japanese, Javanese, Kannada, Kapampangan, Kazakh, Khmer, Kirghiz, Korean, Kurdish (Kurmanji), Kurdish (Sorani), Latin, Latvian, Limburgish, Lithuanian, Lombard, Low Saxon, Luxembourgish, Macedonian, Maithili, Malagasy, Malay, Malayalam, Maltese, Manx, Marathi, Mazandarani, Meadow Mari, Minangkabau, Mingrelian, Mirandese, Mongolian, Nahuatl, Neapolitan, Nepali, Newar, North Frisian, Northern Sotho, Norwegian (Bokmål), Norwegian (Nynorsk), Occitan, Oriya, Ossetian, Palatinate German, Pashto, Persian, Piedmontese, Polish, Portuguese, Quechua, Romanian, Romansh, Russian, Sakha, Sanskrit, Sardinian, Scots, Scottish Gaelic, Serbian, Serbo-Croatian, Sicilian, Sindhi, Sinhalese, Slovak, Slovenian, Somali, Southern Azerbaijani, Spanish, Sundanese, Swahili, Swedish, Tagalog, Tajik, Tamil, Tatar, Telugu, Thai, Tibetan, Turkish, Turkmen, Ukrainian, Upper Sorbian, Urdu, Uyghur, Uzbek, Venetian, Vietnamese, Volapük, Walloon, Waray, Welsh, West Flemish, West Frisian, Western Punjabi, Yiddish, Yoruba, Zazaki, Zeelandic

(157 languages)

### Wikipedia word vectors

**Link:** https://fasttext.cc/docs/en/pretrained-vectors.html

**Algorithm**

Skip-gram model described in Bojanowski et al [https://arxiv.org/abs/1607.04606] with default parameters

**Training data**

- Wikipedia

As mentioned before, Wikipedia is the largest free online encyclopedia, available in more than 200 different languages; because the articles are curated, the corresponding text is of high quality, making Wikipedia a great resource for (multilingual) NLP.

**Languages**

Abkhazian, Acehnese, Adyghe, Afar, Afrikaans, Akan, Albanian, Alemannic, Amharic, Anglo-Saxon, Arabic, Aragonese, Aramaic, Armenian, Aromanian, Assamese, Asturian, Avar, Aymara, Azerbaijani, Bambara, Banjar, Banyumasan, Bashkir, Basque, Bavarian, Belarusian, Bengali, Bihari, Bishnupriya Manipuri, Bislama, Bosnian, Breton, Buginese, Bulgarian, Burmese, Buryat, Cantonese, Catalan, Cebuano, Central Bicolano, Chamorro, Chavacano, Chechen, Cherokee, Cheyenne, Chichewa, Chinese, Choctaw, Chuvash, Classical Chinese, Cornish, Corsican, Cree, Crimean Tatar, Croatian, Czech, Danish, Divehi, Dutch, Dutch Low Saxon, Dzongkha, Eastern Punjabi, Egyptian Arabic, Emilian-Romagnol, English, Erzya, Esperanto, Estonian, Ewe, Extremaduran, Faroese, Fiji Hindi, Fijian, Finnish, Franco-Provençal, French, Friulian, Fula, Gagauz, Galician, Gan, Georgian, German, Gilaki, Goan Konkani, Gothic, Greek, Greenlandic, Guarani, Gujarati, Haitian, Hakka, Hausa, Hawaiian, Hebrew, Herero, Hill Mari, Hindi, Hiri Motu, Hungarian, Icelandic, Ido, Igbo, Ilokano, Indonesian, Interlingua, Interlingue, Inuktitut, Inupiak, Irish, Italian, Jamaican Patois, Japanese, Javanese, Kabardian, Kabyle, Kalmyk, Kannada, Kanuri, Kapampangan, Karachay-Balkar, Karakalpak, Kashmiri, Kashubian, Kazakh, Khmer, Kikuyu, Kinyarwanda, Kirghiz, Kirundi, Komi, Komi-Permyak, Kongo, Korean, Kuanyama, Kurdish (Kurmanji), Kurdish (Sorani), Ladino, Lak, Lao, Latgalian, Latin, Latvian, Lezgian, Ligurian, Limburgish, Lingala, Lithuanian, Livvi-Karelian, Lojban, Lombard, Low Saxon, Lower Sorbian, Luganda, Luxembourgish, Macedonian, Maithili, Malagasy, Malay, Malayalam, Maltese, Manx, Maori, Marathi, Marshallese, Mazandarani, Meadow Mari, Min Dong, Min Nan, Minangkabau, Mingrelian, Mirandese, Moksha, Moldovan, Mongolian, Muscogee, Nahuatl, Nauruan, Navajo, Ndonga, Neapolitan, Nepali, Newar, Norfolk, Norman, North Frisian, Northern Luri, Northern Sami, Northern Sotho, Norwegian (Bokmål), Norwegian (Nynorsk), Novial, Nuosu, Occitan, Old Church Slavonic, Oriya, Oromo, Ossetian, Palatinate German, Pali, Pangasinan, Papiamentu, Pashto, Pennsylvania German, Persian, Picard, Piedmontese, Polish, Pontic, Portuguese, Quechua, Ripuarian, Romani, Romanian, Romansh, Russian, Rusyn, Sakha, Samoan, Samogitian, Sango, Sanskrit, Sardinian, Saterland Frisian, Scots, Scottish Gaelic, Serbian, Serbo-Croatian, Sesotho, Shona, Sicilian, Silesian, Simple English, Sindhi, Sinhalese, Slovak, Slovenian, Somali, Southern Azerbaijani, Spanish, Sranan, Sundanese, Swahili, Swati, Swedish, Tagalog, Tahitian, Tajik, Tamil, Tarantino, Tatar, Telugu, Tetum, Thai, Tibetan, Tigrinya, Tok Pisin, Tongan, Tsonga, Tswana, Tulu, Tumbuka, Turkish, Turkmen, Tuvan, Twi, Udmurt, Ukrainian, Upper Sorbian, Urdu, Uyghur, Uzbek, Venda, Venetian, Vepsian, Vietnamese, Volapük, Võro, Walloon, Waray, Welsh, West Flemish, West Frisian, Western Punjabi, Wolof, Wu, Xhosa, Yiddish, Yoruba, Zazaki, Zeelandic, Zhuang, Zulu

(294 languages)

### 1. Download model

Download the model directly by following the link and downloading the binary or text file or run the following code

In [ ]:
import fasttext.util
fasttext.util.download_model('zh', if_exists='ignore')

Note: 'zh' is abbreviation for Chinese.

Other abbreviations:

| Language | Abbreviations | Language | Abbreviations | Language | Abbreviations | Language | Abbreviations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Afrikaans | af | Fiji Hindi | hif | Malagasy | mg | Sicilian | scn |
| Albanian | sq | Finnish | fi | Malay | ms | Sindhi | sd |
| Alemannic | als | French | fr | Malayalam | ml | Sinhalese | si |
| Amharic | am | Galician | gl | Maltese | mt | Slovak | sk |
| Arabic | ar | Georgian | ka | Manx | gv | Slovenian | sl |
| Aragonese | an | German | de | Marathi | mr | Somali | so |
| Armenian | hy | Goan Konkani | gom | Mazandarani | mzn | Southern Azerbaijani | azb |
| Assamese | as | Greek | el | Meadow Mari | mhr | Spanish | es |
| Asturian | ast | Gujarati | gu | Minangkabau | min | Sundanese | su |
| Azerbaijani  | az | Haitian | ht | Mingrelian | xmf | Swahili | sw |
| Bashkir | ba | Hebrew | he | Mirandese | mwl | Swedish | sv |
| Basque  | eu | Hill Mari | mrj | Mongolian | mn | Tagalog | tl |
| Bavarian  | bar | Hindi | hi | Nahuatl | nah | Tajik | tg |
| Belarusian | be | Hungarian | hu | Neapolitan | nap | Tamil | ta |
| Bengali | bn | Icelandic | is | Nepali | ne | Tatar | tt |
| Bihari | bh | Ido | io | Newar | new | Telugu | te |
| Bishnupriya Manipuri | bpy | Ilokano | ilo | North Frisian | frr | Thai | th |
| Bosnian | bs | Indonesian | id | Northern Sotho | nso | Tibetan | bo |
| Breton | br | Interlingua | ia | Norwegian (Bokmål) | no | Turkish | tr |
| Bulgarian  | bg | Irish | ga | Norwegian (Nynorsk) | nn | Turkmen | tk |
| Burmese | my | Italian | it | Occitan | oc | Ukrainian | uk |
| Catalan | ca | Japanese | ja | Oriya | or | Upper Sorbian | hsb |
| Cebuano | ceb | Javanese | jv | Ossetian | os | Urdu | ur |
| Central Bicolano | bcl | Kannada | kn | Palatinate German | pfl | Uyghur | ug |
| Chechen | ce | Kapampangan | pam | Pashto | ps | Uzbek | uz |
| Chinese | zh | Kazakh | kk | Persian | fa | Venetian | vec |
| Chuvash | cv | Khmer | km | Piedmontese | pms | Vietnamese | vi |
| Corsican | co | Kirghiz | ky | Polish | pl | Volapük | vo |
| Croatian | hr | Korean | ko | Portuguese | pt | Walloon | wa |
| Czech  | cs | Kurdish (Kurmanji) | ku | Quechua | qu | Waray | war |
| Danish | da | Kurdish (Sorani) | ckb | Romanian | ro | Welsh | cy |
| Divehi | dv | Latin | la | Romansh | rm | West Flemish | vls |
| Dutch | nl | Latvian | lv | Russian | ru | West Frisian | fy |
| Eastern Punjabi | pa | Limburgish | li | Sakha | sah | Western Punjabi | pnb |
| Egyptian Arabic | arz | Lithuanian | lt | Sanskrit | sa | Yiddish | yi |
| Emilian-Romagnol | emn | Lombard | lmo | Sardinian | sc | Yoruba | yo |
| English | en | Low Saxon | nds | Scots | sco | Zazaki | diq |
| Erzya | myv | Luxembourgish | lb | Scottish Gaelic | gcl | Zeelandic | zea |
| Esperanto | eo | Macedonian | mk | Serbian | sr |
| Estonian | et | Maithili | mal | Serbo-Croatian | sh |

### 2. Load model

In [ ]:
model = fasttext.load_model('cc.zh.300.bin')

### 3. Reduce the dimension of the vectors (optional)

In [ ]:
model.get_dimension()
required_dimension = 100
fasttext.util.reduce_model(model, required_dimension)
model.get_dimension()

### 4. Get dictionary

In [ ]:
dictionary = {key : model.get_word_vector(key) for key in model.get_words()}

For checking, the dictionary contains 2000000 different words

In [ ]:
len(dictionary)

### 5. Save dictionary in file

In [ ]:
save_dictionary('chinese_fasttext_dictionary.txt', dictionary, (len(dictionary), model.get_dimension()))

## Results

https://shortlinks.plus/lswRq

This repository contains:
- corpuses used to train models for Russian, English, German, Vietnamese, Armenian, Sinhala, Tatar (\<language\>_lit_corpus.txt)
- obtained dictionaries for each model trained on the corpus above (\<language\>_lit_dictionary_cbow.txt and \<language\>_lit_dictionary_skipgram.txt)
- obtained dictionaries for pre-trained model (\<language\>_fasttext_dictionary.txt)

The dictionary file is structured as the following:
- the first line contains two space-separated integers: *n* number of words in the dictionary and *d* dimension of the vector
- in the next *n* lines there is a word and after a space a set of *d* space-separated fractional numbers, which is the vector representation of the word

**<center>Number of words in the obtained dictionaries<center>**

| Language | Our corpus (CBOW) | Our corpus (Skipgram) | Fasttext |
| --- | --- | --- | --- |
| Armenian | <center>117 688<center> | <center>117 688<center> | 2 000 000 |
| English | <center>315 482<center> | <center>315 482<center> | 2 000 000 |
| German | <center>1 066 101<center> | <center>1 066 101<center> | 2 000 000 |
| Russian | <center>192 881<center> | <center>192 881<center> | 2 000 000 |
| Sinhala | <center>232 338<center> | <center>232 338<center> | 2 000 000 |
| Tatar | <center><center>423 265<center> | <center>423 265<center> | 2 000 000 |
| Vietnamese | <center>51 977<center> | <center>51 977<center> | 2 000 000 |